# 1 · Data model and SQL
Normalised SQLite schema: `users`, `interests`, `user_interests`, `connections`, plus `places` and `checkins`
for the secondary "met at the same coffee shop" signal. Data is synthetic: 200 users from 8 hidden personas.
**Phone and email are stored but never used as features or returned by the recommender.**

In [1]:
import pandas as pd
from recsys import db
from recsys.models import Recommender, DEFAULT_WEIGHTS
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

path = db.build_db()          # recreates data/recsys.db deterministically (seed 42)
conn = db.connect(path)
pd.read_sql_query("SELECT name AS tbl, (SELECT COUNT(*) FROM pragma_table_info(name)) AS columns FROM sqlite_master WHERE type='table'", conn)

,tbl,columns
0,users,0
1,places,0
2,checkins,0
3,interests,0
4,user_interests,0
5,connections,0


In [2]:
pd.read_sql_query("SELECT 'users' t, COUNT(*) n FROM users UNION ALL SELECT 'interests', COUNT(*) FROM interests "
                  "UNION ALL SELECT 'user_interests', COUNT(*) FROM user_interests UNION ALL SELECT 'connections', COUNT(*) FROM connections "
                  "UNION ALL SELECT 'places', COUNT(*) FROM places UNION ALL SELECT 'checkins', COUNT(*) FROM checkins", conn)

,t,n
0,users,200
1,interests,52
2,user_interests,1144
3,connections,779
4,places,18
5,checkins,394


In [3]:
pd.read_sql_query("SELECT * FROM users LIMIT 5", conn)

,id,name,phone,email,created_at,last_active_at
0,1,Isha Sharma,+91-90000-00001,isha.sharma1@example.com,2026-11-20 15:13:23,2026-10-24
1,2,Suresh Menon,+91-90000-00002,suresh.menon2@example.com,2026-11-20 15:13:23,2026-11-09
2,3,Siddharth Iyer,+91-90000-00003,siddharth.iyer3@example.com,2026-11-20 15:13:23,2026-10-25
3,4,Arjun Bhat,+91-90000-00004,arjun.bhat4@example.com,2026-11-20 15:13:23,2026-11-14
4,5,Harsha Kulkarni,+91-90000-00005,harsha.kulkarni5@example.com,2026-11-20 15:13:23,2026-08-21


### How common is each interest?
The four generic ones (music, movies, friends, social media) are everywhere: that is exactly what TF-IDF down-weights.

In [4]:
pd.read_sql_query("""SELECT i.label, COUNT(*) AS users FROM user_interests ui JOIN interests i ON i.id = ui.interest_id
GROUP BY i.label ORDER BY users DESC""", conn).head(12)

,label,users
0,watching movies,113
1,scrolling social media,103
2,hanging out with friends,102
3,listening to music,101
4,book club meetups,21
5,journaling,21
6,collecting vinyl records,20
7,watching sci-fi movies,20
8,playing dungeons and dragons,19
9,trying street food,19


### Retrieval query: one row per user with aggregated interests

In [5]:
print(db.PROFILES_SQL)
profiles = db.load_profiles(conn)
profiles.head()


SELECT u.id AS user_id, u.name, GROUP_CONCAT(i.label, '|') AS interests
FROM users u
JOIN user_interests ui ON ui.user_id = u.id
JOIN interests i       ON i.id = ui.interest_id
GROUP BY u.id, u.name
ORDER BY u.id



,user_id,name,interests
0,1,Isha Sharma,"[astronomy and stargazing, learning the piano, marvel movie nights, space documentarie..."
1,2,Suresh Menon,"[backpacking trips, hanging out with friends, reading science fiction novels, scrollin..."
2,3,Siddharth Iyer,"[morning yoga, space documentaries, watching movies, watching sci-fi movies]"
3,4,Arjun Bhat,"[marvel movie nights, reading science fiction novels, scrolling social media, star tre..."
4,5,Harsha Kulkarni,"[astronomy and stargazing, collecting vinyl records, hanging out with friends, reading..."


### Exclusion query: who may user 9 be recommended? (not themself, not existing connections)

In [6]:
print(db.CANDIDATES_SQL)
target = 9
friends = pd.read_sql_query("SELECT * FROM connections WHERE user_a = ? OR user_b = ?", conn, params=(target, target))
cands = db.candidate_ids(conn, target)
print("existing connections:", friends.values.tolist())
friend_ids = set(friends.values.ravel()) - {target}
print(f"{len(cands)} candidates; self excluded: {target not in cands}; friends excluded: {not friend_ids & set(cands)}")


SELECT u.id AS user_id
FROM users u
WHERE u.id != :target
  AND NOT EXISTS (
      SELECT 1 FROM connections c
      WHERE (c.user_a = :target AND c.user_b = u.id)
         OR (c.user_b = :target AND c.user_a = u.id))
ORDER BY u.id

existing connections: [[9, 14, '2026-11-20 15:13:23'], [9, 18, '2026-11-20 15:13:23'], [9, 60, '2026-11-20 15:13:23'], [9, 74, '2026-11-20 15:13:23'], [9, 96, '2026-11-20 15:13:23'], [9, 128, '2026-11-20 15:13:23'], [9, 135, '2026-11-20 15:13:23'], [9, 166, '2026-11-20 15:13:23'], [8, 9, '2026-11-20 15:13:23']]
190 candidates; self excluded: True; friends excluded: True


### Secondary signals: places visited in the last 90 days, days since last active

In [7]:
db.load_signals(conn).head(10)

,user_id,days_inactive,places
0,1,27,"[Dice & Dragons Board Game Cafe, Phoenix City Mall]"
1,2,11,"[Orbit Comics & Games, Jayanagar, Phoenix City Mall]"
2,3,26,"[Maker Space BLR, Orbit Comics & Games, Jayanagar, Planetarium Night Shows]"
3,4,6,"[Orbit Comics & Games, Jayanagar, Phoenix City Mall, Planetarium Night Shows]"
4,5,91,"[Orbit Comics & Games, Jayanagar, Planetarium Night Shows]"
5,6,43,"[IronCore Gym, HSR, Orbit Comics & Games, Jayanagar, Planetarium Night Shows]"
6,7,19,"[Orbit Comics & Games, Jayanagar, Phoenix City Mall, Planetarium Night Shows]"
7,8,71,[Phoenix City Mall]
8,9,2,"[IronCore Gym, HSR, Planetarium Night Shows]"
9,10,24,"[Phoenix City Mall, Planetarium Night Shows]"
